# Data Generation v3: Spec-Based (Refined)

**Method B refined**: LLM generates spec → Python generates rows
- Fewer fixture rows (max 10 per table)
- Parallel spec generation for independent tables
- More programmatic rows (300+ for fact tables)
- Better distribution support

In [ ]:
%pip install openai numpy
dbutils.library.restartPython()

In [ ]:
# === INPUTS ===

COMPANY_NAME = "First National Bank"
COMPANY_DESCRIPTION = "First National Bank is a regional community bank operating across the southeastern United States with 45 branches. They offer personal and business banking, mortgages, auto loans, and wealth management services. They serve approximately 200,000 customers."

MUST_ANSWER_QUESTIONS = [
    "What is the total loan portfolio value by loan type?",
    "Which branch has the highest deposit growth this year?",
    "What is the average interest rate by loan category?",
    "How many new accounts were opened per month?",
    "What is the customer distribution by account type?",
    "Which region has the highest default rate?",
]

DATABRICKS_HOST_ID = "7474655921234161"
LLM_MODEL = "opendoor-claude-opus-46"

import json, re, time, random, datetime
import numpy as np
from concurrent.futures import ThreadPoolExecutor, as_completed
from openai import OpenAI
from pyspark.sql.types import *

ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
token = ctx.apiToken().get()

client = OpenAI(
    api_key=token,
    base_url=f"https://{DATABRICKS_HOST_ID}.ai-gateway.cloud.databricks.com/mlflow/v1",
)

def call_llm(system_prompt, user_prompt, max_tokens=8192):
    resp = client.chat.completions.create(
        model=LLM_MODEL, max_tokens=max_tokens,
        messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}],
    )
    return resp.choices[0].message.content.strip()

def parse_json(raw):
    if raw.startswith('```'):
        raw = raw.split('\n', 1)[1]
        if raw.endswith('```'): raw = raw[:raw.rfind('```')]
    for sc, ec in [('{', '}'), ('[', ']')]:
        s, e = raw.find(sc), raw.rfind(ec) + 1
        if s >= 0 and e > s:
            c = re.sub(r',\s*([}\]])', r'\1', raw[s:e])
            try: return json.loads(c)
            except: continue
    return json.loads(raw)

questions_text = '\n'.join(f'- {q}' for q in MUST_ANSWER_QUESTIONS)
print(f'Company: {COMPANY_NAME}')
print(f'Questions: {len(MUST_ANSWER_QUESTIONS)}')
print('Ready.')

---
## Step 1: Schema Design

In [ ]:
SCHEMA_PROMPT = (
    'You are a data architect. Design a database schema for the given company.\n'
    'Output ONLY valid JSON, no markdown fences.\n\n'
    'RULES:\n'
    '1. Create 3-4 tables representing core business entities.\n'
    '2. Each table: 5-10 columns, snake_case names.\n'
    '3. First column = primary key (sequential integer).\n'
    '4. Use FKs to link tables. Specify references as table.column in the references field.\n'
    '5. Every table needs at least one DATE and one numeric column.\n'
    '6. Column comments are CRITICAL — passed to the AI query engine.\n'
    '7. Tables ordered: referenced BEFORE referencing.\n'
    '8. MUST-ANSWER QUESTIONS: Design tables that can answer ALL of these via SQL.\n'
    '9. Dimension tables: 25-40 rows. Fact tables: 300-500 rows.\n\n'
    'SQL TYPES: STRING, INT, DOUBLE, DATE, BOOLEAN, BIGINT\n\n'
    'OUTPUT FORMAT: A JSON object with "tables" array and "sample_questions" array.\n'
    'Each table has: name, comment, row_count, table_type (dimension or fact), columns array.\n'
    'Each column has: name, type, comment, primary_key (boolean), references (empty string or table.column).'
)

print('Designing schema...')
t_total = time.time()
t0 = time.time()
schema = parse_json(call_llm(SCHEMA_PROMPT, f'Company: {COMPANY_NAME}\nDescription: {COMPANY_DESCRIPTION}\n\nMUST-ANSWER QUESTIONS:\n{questions_text}\n\nDesign the schema.'))
t_schema = time.time() - t0

print(f'Schema: {t_schema:.1f}s')
for t in schema['tables']:
    print(f"  {t['name']} ({t.get('table_type','?')}, {t.get('row_count','?')} rows, {len(t['columns'])} cols)")


---
## Step 2: Generate Specs (Parallel for Independent Tables)

In [ ]:
# === Row generator from spec (define before running Step 2) ===
# NOTE: Run this cell BEFORE Step 2!

def generate_rows(spec, row_count, generated_tables):
    """Generate rows from an LLM-designed spec."""
    columns = spec.get('columns', {})
    fixtures = spec.get('fixtures', [])
    
    rows = list(fixtures)  # Copy fixtures
    remaining = max(0, row_count - len(rows))
    start_id = len(rows) + 1
    
    for i in range(remaining):
        row = {}
        for col_name, cs in columns.items():
            dist = cs.get('dist', 'fixed')
            try:
                if dist == 'sequential':
                    row[col_name] = start_id + i
                elif dist == 'fk_sample':
                    parent = generated_tables.get(cs.get('from_table', ''), [])
                    ids = [r.get(cs.get('from_column', '')) for r in parent if r.get(cs.get('from_column', '')) is not None]
                    row[col_name] = random.choice(ids) if ids else random.randint(1, 10)
                elif dist == 'weighted_choice':
                    vals = cs.get('values', ['A'])
                    wts = cs.get('weights', [1.0/len(vals)] * len(vals))
                    total = sum(wts)
                    wts = [w/total for w in wts]
                    row[col_name] = random.choices(vals, weights=wts, k=1)[0]
                elif dist == 'uniform_int':
                    row[col_name] = random.randint(cs.get('min', 0), cs.get('max', 100))
                elif dist == 'uniform_float':
                    row[col_name] = round(random.uniform(cs.get('min', 0), cs.get('max', 1000)), cs.get('decimals', 2))
                elif dist == 'normal':
                    v = np.random.normal(cs.get('mean', 50), cs.get('std', 10))
                    v = max(cs.get('min', 0), min(cs.get('max', 1e9), v))
                    row[col_name] = round(float(v), cs.get('decimals', 2))
                elif dist == 'date_range':
                    s = datetime.date.fromisoformat(cs.get('start', '2023-01-01'))
                    e = datetime.date.fromisoformat(cs.get('end', '2025-04-28'))
                    d = (e - s).days
                    row[col_name] = (s + datetime.timedelta(days=random.randint(0, max(d, 1)))).isoformat()
                elif dist == 'boolean':
                    row[col_name] = random.random() < cs.get('true_pct', 0.5)
                elif dist == 'formula':
                    try:
                        v = eval(cs.get('expr', '0'), {'__builtins__': {}, 'round': round, 'max': max, 'min': min}, row)
                        row[col_name] = round(v, 2) if isinstance(v, float) else v
                    except: row[col_name] = 0
                elif dist == 'fixed':
                    row[col_name] = cs.get('value', '')
                else:
                    row[col_name] = None
            except Exception:
                row[col_name] = None
        rows.append(row)
    
    return rows[:row_count]

print('Row generator ready. Now run Step 2.')

SPEC_PROMPT = (
    'You are a data engineer designing a data generation specification.\n'
    'Output ONLY valid JSON, no markdown fences.\n\n'
    'For EACH column, specify one of these distribution types:\n'
    '- sequential: start from 1 — for primary keys\n'
    '- fk_sample: from_table + from_column — samples from parent table\n'
    '- weighted_choice: values list + weights list — categorical with realistic distribution\n'
    '- uniform_int: min + max — random integer in range\n'
    '- uniform_float: min + max + decimals — random float in range\n'
    '- normal: mean + std + min + max + decimals — normal distribution with clipping\n'
    '- date_range: start + end — random date in range (ISO format)\n'
    '- boolean: true_pct — random boolean with given probability\n'
    '- formula: expr — derived from other columns in the same row\n\n'
    'IMPORTANT RULES FOR FIXTURES:\n'
    '- Include 5-10 fixture rows MAX for fact tables.\n'
    '- Fixtures are specific rows that MUST exist to answer the must-answer questions.\n'
    '- For dimension tables, fixtures ARE the primary data (all rows are meaningful).\n'
    '- For fact tables, most data comes from distributions, not fixtures.\n\n'
    'IMPORTANT RULES FOR DISTRIBUTIONS:\n'
    '- weighted_choice values must be REAL and domain-specific.\n'
    '- Numeric distributions must be realistic for the domain.\n\n'
    'OUTPUT: JSON object with columns dict (col_name -> dist spec) and fixtures array (row objects).'
)

def get_deps(table_def):
    deps = set()
    for c in table_def.get('columns', []):
        ref = c.get('references', '')
        if ref: deps.add(ref.split('.')[0] if '.' in ref else ref)
    return deps

def gen_spec(table_def, generated_tables):
    name = table_def['name']
    col_lines = []
    for c in table_def['columns']:
        d = f"- {c['name']} ({c.get('type','STRING')})"
        if c.get('comment'): d += f": {c['comment']}"
        if c.get('primary_key'): d += ' [PK]'
        if c.get('references'): d += f" [FK -> {c['references']}]"
        col_lines.append(d)
    
    fk_info = []
    for c in table_def['columns']:
        ref = c.get('references', '')
        if not ref: continue
        rt = ref.split('.')[0] if '.' in ref else ref
        rc = ref.split('.')[1] if '.' in ref else f'{ref}_id'
        parent = generated_tables.get(rt, [])
        if parent:
            ids = list(set(str(r.get(rc)) for r in parent if r.get(rc) is not None))[:40]
            fk_info.append(f"Available {c['name']} values: {ids}")
    
    col_text = '\n'.join(col_lines)
    fk_text = '\n'.join(fk_info) if fk_info else ''
    
    prompt = (
        f'Company: {COMPANY_NAME}\n'
        f'Description: {COMPANY_DESCRIPTION}\n'
        f'Table: "{name}" — {table_def.get("comment","")}\n'
        f'Row count: {table_def.get("row_count", 100)} | Type: {table_def.get("table_type", "fact")}\n\n'
        f'Columns:\n{col_text}\n'
        f'{fk_text}\n\n'
        f'Must-answer questions:\n{questions_text}\n\n'
        f'Design the spec. MAX 10 fixture rows for fact tables.'
    )
    
    for attempt in range(3):
        try:
            raw = call_llm(SPEC_PROMPT, prompt)
            return parse_json(raw)
        except Exception as e:
            if attempt == 2:
                print(f'  [{name}] SPEC FAILED: {str(e)[:80]}')
                return {'columns': {}, 'fixtures': []}
            print(f'  [{name}] Spec parse retry {attempt+1}')

# Build dependency levels
tables_by_name = {t['name']: t for t in schema['tables']}
levels, resolved, remaining = [], set(), set(tables_by_name.keys())
while remaining:
    current = [n for n in remaining if get_deps(tables_by_name[n]).issubset(resolved)]
    if not current: current = list(remaining)
    levels.append(current)
    resolved.update(current)
    remaining -= set(current)

print(f'Dependency levels: {levels}')

# Generate specs level by level (parallel within level)
t0 = time.time()
specs = {}
generated_data = {}

for level_idx, level_names in enumerate(levels):
    parallel = len(level_names) > 1
    print(f'\n=== Level {level_idx}: {level_names} {"(parallel)" if parallel else ""} ===')
    
    if not parallel:
        name = level_names[0]
        st = time.time()
        spec = gen_spec(tables_by_name[name], generated_data)
        specs[name] = spec
        print(f'  [{name}] Spec in {time.time()-st:.1f}s | {len(spec.get("fixtures",[]))} fixtures')
        rows = generate_rows(spec, tables_by_name[name].get('row_count', 100), generated_data)
        generated_data[name] = rows
        print(f'  [{name}] {len(rows)} rows generated')
    else:
        def _gen(n):
            st = time.time()
            spec = gen_spec(tables_by_name[n], generated_data)
            return n, spec, time.time()-st
        
        with ThreadPoolExecutor(max_workers=min(len(level_names), 4)) as ex:
            futures = {ex.submit(_gen, n): n for n in level_names}
            for f in as_completed(futures):
                name, spec, dur = f.result()
                specs[name] = spec
                print(f'  [{name}] Spec in {dur:.1f}s | {len(spec.get("fixtures",[]))} fixtures')
        
        for name in level_names:
            rows = generate_rows(specs[name], tables_by_name[name].get('row_count', 100), generated_data)
            generated_data[name] = rows
            print(f'  [{name}] {len(rows)} rows generated')

t_specs = time.time() - t0
print(f'\nTotal spec+gen time: {t_specs:.1f}s')


---
## Step 3: Evaluate Results

In [ ]:
# === TIMING SUMMARY ===
total_time = time.time() - t_total
print('=' * 60)
print('TIMING SUMMARY')
print('=' * 60)
print(f'Schema design:     {t_schema:.1f}s')
print(f'Specs + row gen:   {t_specs:.1f}s')
print(f'TOTAL:             {total_time:.1f}s')
print()
print('Row counts:')
for name, rows in generated_data.items():
    ttype = tables_by_name[name].get('table_type', '?')
    n_fix = len(specs[name].get('fixtures', []))
    print(f'  {name}: {len(rows)} rows ({n_fix} fixtures + {len(rows)-n_fix} generated) [{ttype}]')

In [ ]:
# === SAMPLE ROWS ===
print('=' * 60)
print('SAMPLE ROWS (first 5 per table)')
print('=' * 60)

for name, rows in generated_data.items():
    n_fix = len(specs[name].get('fixtures', []))
    print(f'\n--- {name} ({len(rows)} rows, {n_fix} fixtures) ---')
    print('  Fixture rows:')
    for r in rows[:min(3, n_fix)]:
        print(f'    {r}')
    if len(rows) > n_fix:
        print('  Generated rows:')
        for r in rows[n_fix:n_fix+3]:
            print(f'    {r}')

In [ ]:
# === DATA QUALITY CHECKS ===
print('=' * 60)
print('DATA QUALITY CHECKS')
print('=' * 60)

for name, rows in generated_data.items():
    if not rows: continue
    print(f'\n--- {name} ---')
    cols = list(rows[0].keys())
    
    for col in cols:
        vals = [r.get(col) for r in rows if r.get(col) is not None]
        if not vals: continue
        
        # Numeric check
        try:
            nums = [float(v) for v in vals]
            print(f'  {col}: min={min(nums):.1f}, max={max(nums):.1f}, avg={sum(nums)/len(nums):.1f}, unique={len(set(nums))}')
            continue
        except (ValueError, TypeError):
            pass
        
        # Categorical check
        str_vals = [str(v) for v in vals]
        unique = set(str_vals)
        if len(unique) <= 20:
            from collections import Counter
            counts = Counter(str_vals).most_common(10)
            dist_str = ', '.join(f'{v}:{c}' for v, c in counts)
            print(f'  {col}: {len(unique)} unique — {dist_str}')
        else:
            print(f'  {col}: {len(unique)} unique — sample: {list(unique)[:5]}')

In [ ]:
# === MUST-ANSWER QUESTION VERIFICATION ===
print('=' * 60)
print('MUST-ANSWER QUESTION VERIFICATION')
print('=' * 60)

for q in MUST_ANSWER_QUESTIONS:
    print(f'\nQ: {q}')
    # Try to identify which tables/columns are relevant
    q_lower = q.lower()
    for name, rows in generated_data.items():
        if not rows: continue
        cols = list(rows[0].keys())
        relevant_cols = [c for c in cols if any(w in c for w in q_lower.split() if len(w) > 3)]
        if relevant_cols:
            print(f'  → {name}: relevant columns = {relevant_cols}')
            # Show sample values for relevant columns
            for rc in relevant_cols[:2]:
                sample_vals = set(str(r.get(rc, ''))[:30] for r in rows[:20])
                print(f'    {rc} samples: {list(sample_vals)[:5]}')

print('\n\nFixture summary:')
for name, spec in specs.items():
    n = len(spec.get('fixtures', []))
    print(f'  {name}: {n} fixture rows')

---
## Step 4: Write to Delta Tables (Optional — run if you want to test in Genie)

In [ ]:
# === OPTIONAL: Write to Delta tables for Genie testing ===
# Uncomment and run if you want to create actual tables

# CATALOG = 'yd_launchpad_final_classic_catalog'
# SCHEMA_NAME = 'genie_app'
# COMPANY_SLUG = re.sub(r'[^a-zA-Z0-9]+', '_', COMPANY_NAME.lower()).strip('_')[:50]
# 
# TYPE_MAP = {
#     'STRING': StringType(), 'INT': IntegerType(), 'DOUBLE': DoubleType(),
#     'DATE': DateType(), 'BOOLEAN': BooleanType(), 'BIGINT': LongType(),
# }
# 
# for tdef in schema['tables']:
#     name = tdef['name']
#     rows = generated_data.get(name, [])
#     if not rows: continue
#     
#     prefixed = f'{COMPANY_SLUG}_{name}'
#     full_name = f'`{CATALOG}`.`{SCHEMA_NAME}`.`{prefixed}`'
#     
#     fields = [StructField(c['name'], TYPE_MAP.get(c.get('type','STRING'), StringType()), True) for c in tdef['columns']]
#     spark_schema = StructType(fields)
#     
#     # Coerce types
#     clean = []
#     for row in rows:
#         cr = {}
#         for col, field in zip(tdef['columns'], fields):
#             v = row.get(col['name'])
#             if v is None: cr[col['name']] = None
#             elif isinstance(field.dataType, (IntegerType, LongType)): cr[col['name']] = int(v) if v else None
#             elif isinstance(field.dataType, DoubleType): cr[col['name']] = float(v) if v else None
#             elif isinstance(field.dataType, BooleanType): cr[col['name']] = bool(v) if isinstance(v, bool) else str(v).lower() in ('true','1')
#             elif isinstance(field.dataType, DateType):
#                 try: cr[col['name']] = datetime.date.fromisoformat(str(v)[:10])
#                 except: cr[col['name']] = None
#             else: cr[col['name']] = str(v)
#         clean.append(cr)
#     
#     spark.sql(f'DROP TABLE IF EXISTS {full_name}')
#     df = spark.createDataFrame(clean, spark_schema)
#     df.write.saveAsTable(f'{CATALOG}.{SCHEMA_NAME}.{prefixed}')
#     print(f'{full_name}: {len(clean)} rows')
# 
# print('Done! Tables ready for Genie.')